# mievformer Tutorial

This tutorial demonstrates the workflow of `mievformer` package for nicheformer analysis.

## 1. Setup and Data Loading

First, we load the necessary libraries and the dataset. We will use a subsampled version of the lung dataset for this tutorial.

In [ ]:
import mievformer as mf
import scanpy as sc
import numpy as np
import os

# Path to the original data
original_data_path = "nichedynamics/data/20230629__230629pre/output-XETG00057__0003908__lung__20230629__073037/adata.h5ad"

# Load data
if os.path.exists(original_data_path):
    adata = sc.read_h5ad(original_data_path)
    print(f"Loaded data with {adata.shape[0]} cells and {adata.shape[1]} genes.")
    
    # Subsample to 10k cells for tutorial
    if adata.shape[0] > 10000:
        sc.pp.subsample(adata, n_obs=10000)
        print(f"Subsampled to {adata.shape[0]} cells.")
else:
    print(f"Data file not found at {original_data_path}. Please check the path.")
    # Create dummy data for testing if file not found
    adata = sc.AnnData(np.random.rand(10000, 100))
    adata.obsm['spatial'] = np.random.rand(10000, 2)
    adata.obsm['X_pca'] = np.random.rand(10000, 20)
    adata.obs['cell_type'] = np.random.choice(['TypeA', 'TypeB', 'TypeC'], 10000)
    print("Created dummy data.")


## 2. Optimize NicheFormer Model

We optimize the NicheFormer model to learn the niche representation. This step involves training a model to reconstruct the cellular neighborhood.

In [ ]:
# Define model path
model_path = "tutorial_model.pth"

# Optimize model
# We use a small number of epochs for demonstration purposes
adata = mf.optimize_nicheformer(
    adata, 
    model_path=model_path,
    max_epochs=10,  # Increase for real analysis
    batch_size=128,
    latent_dim=20,
    neighbor_num=100
)

print("Optimization complete.")
print("Added keys to adata.obsm:", adata.obsm.keys())

## 3. Calculate wb_ez

Calculate the weight and bias terms for the embedding.

In [ ]:
# Note: In the current implementation, optimize_nicheformer already adds 'e' (embedding) to adata.
# calculate_wb_ez is a placeholder for more advanced analysis if needed, 
# or if loading a pre-trained model.
# For this tutorial, we can skip explicit call if optimize_nicheformer did the job, 
# or we can demonstrate it if implemented.

# mf.calculate_wb_ez(adata, model_path=model_path)
print("Embedding 'e' shape:", adata.obsm['e'].shape)

## 4. Spatial Distribution and Aggregation

Calculate the spatial distribution of cells and aggregate the embeddings.

In [ ]:
# Calculate spatial distribution (requires model object, but we saved state_dict)
# For this tutorial, we will reload the model structure to use it.
# In a real scenario, you might keep the model object in memory.

# Re-instantiate model for usage
import torch
from mievformer import nicheformer as nf
from mievformer import workflow as wl

# We need to recreate the dataset to get input dim
ds = wl.adata2ds(adata, neighbor_num=100)
model = nf.NicheFormer(
    input_dim=adata.obsm['nf_cellrep'].shape[1], 
    latent_dim=20, 
    train_ds=ds, 
    val_ds=ds # Dummy
)
model.load_state_dict(torch.load(model_path))

# Calculate spatial distribution
adata = mf.calculate_spatial_distribution(adata, model)

# Aggregate distribution embedding (if applicable)
adata = mf.aggregate_dist_e(adata)
print("Spatial distribution calculated.")

## 5. Estimate Population Density

Estimate the density of specific cell populations.

In [ ]:
# Check available cell types
print(adata.obs['cell_type'].unique())

# Estimate density for a specific group (e.g., the first one found)
target_group = adata.obs['cell_type'].unique()[0]
print(f"Estimating density for: {target_group}")

adata = mf.estimate_population_density(adata, group=target_group, cluster_key='cell_type')

print(f"Density estimated. Added '{target_group}_density' to adata.obs.")

## 6. Analyze Density Correlation

Analyze the correlation between the estimated density and gene expression.

In [ ]:
density_col = f'{target_group}_density'
output_plot = "density_correlation.png"

corrs = mf.analyze_density_correlation(
    adata, 
    density_col=density_col, 
    file_path=output_plot
)

print("Top 5 correlated genes:")
print(corrs.nlargest(5))

# Display the plot
from IPython.display import Image
if os.path.exists(output_plot):
    display(Image(filename=output_plot))

## 7. Analyze Niche Composition

Cluster niches based on their cell type composition and visualize the result.

In [ ]:
# Analyze niche composition
# This function clusters the niches (e.g., 'leiden_e') based on the composition of cell types within them.
# It also generates a clustermap visualization.

# Ensure we have the necessary keys. 'leiden_e' is usually added by optimize_nicheformer or subsequent clustering.
# If not present, we might need to run clustering on 'e' or 'dist_e'.
if 'leiden_e' not in adata.obs:
    print("Running Leiden clustering on 'e' embedding...")
    sc.pp.neighbors(adata, use_rep='e')
    sc.tl.leiden(adata, key_added='leiden_e')

adata = mf.analyze_niche_composition(
    adata,
    n_clusters=3, # Number of meta-clusters for niches
    file_path='niche_composition_clustermap.png'
)

print("Niche composition analysis complete.")
print("Added 'niche_cluster' to obs:", 'niche_cluster' in adata.obs)

# Display the plot
from IPython.display import Image
if os.path.exists('niche_composition_clustermap.png'):
    display(Image(filename='niche_composition_clustermap.png'))